# 02 · Preprocessing — VehicleVision (Classification)

This dataset needs much less preprocessing than the flight-price one: no missing values, no categorical columns, all 20 features numeric (EDA). Two steps remain, and this notebook **measures** why each one matters instead of just asserting it:

| Step | Applied where | Question tested here |
|---|---|---|
| 1. Drop exact duplicate rows | once, before CV (`load_clean()`) | How much do duplicates inflate CV accuracy, and for which models? |
| 2. Standardise features | inside each model's pipeline (per fold) | Which models need it, and what happens to logistic regression without it? |
| 3. Leave clipped values and rounding as they are | — | Why not "fix" the floor spikes found in EDA 2.2/2.7? |

The noise-specific steps (training on noisy copies, noise-aware covariance) are modelling choices and live in `03_models.ipynb`.

**What this cell does:** Standard opening: auto-reload, make `src/` importable, import `config`.

**Why:** Same as every notebook.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

# Make `src` importable: repo root locally, or an uploaded Kaggle dataset containing src/
for p in [Path.cwd(), *Path.cwd().parents, *(f.parent.parent for f in Path("/kaggle/input").glob("**/src/config.py"))]:
    if (p / "src" / "config.py").exists():
        sys.path.insert(0, str(p.resolve()))
        break

from src import config

**What this cell does:** Imports libraries, loads the raw training file (with duplicates) and the cleaned one.

**Why:** Step 1 compares the two directly.

In [2]:
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from src import evaluate
from src.classification import preprocess as pp

TASK = "classification"
pd.set_option("display.width", 160)

raw, _, _ = config.load(TASK)
FEATURES = [c for c in raw.columns if c not in (pp.ID, pp.TARGET)]
X_raw, y_raw = raw[FEATURES], raw[pp.TARGET]
X, y, X_test, test_ids, sample_sub = pp.load_clean()
print("raw train:", X_raw.shape, "| after dropping duplicates:", X.shape)

raw train: (10000, 20) | after dropping duplicates: (7500, 20)


## Step 1 — Duplicates leak into CV

EDA 2.1 found that ids 7501–10000 are exact copies of 2,500 earlier rows (same features, same label).

**What goes wrong if we keep them.** 5-fold CV shuffles the 10,000 rows into folds. For each duplicated pair, the chance that the two copies land in *different* folds is 4/5. Then, when one copy is in the validation fold, its twin is in the training data: the model is "tested" on a row it was trained on. A model that memorises its training rows (a full decision tree, train accuracy 1.000) gets those rows right for free.

**Expected size of the effect:** 5,000 of the 10,000 rows belong to a pair, so ≈ 50% × 4/5 = **40% of validation rows have a twin in training**. A memorising model gets those almost all right, so its CV score should be strongly inflated. A smooth model (logistic regression, train ≈ CV) barely benefits, because it doesn't memorise individual rows.

**What this cell does:** Counts how many validation rows have their twin in the training part, fold by fold, when CV runs on the raw 10,000 rows.

**Why:** Checks the 40% estimate on the actual folds we'd use.

In [3]:
key = X_raw.astype(str).agg("|".join, axis=1).to_numpy()
leaked = []
for tr, va in evaluate._splitter(TASK).split(X_raw, y_raw):
    leaked.append(np.isin(key[va], key[tr]).mean())
print("share of validation rows with an identical row in the training part, per fold:", np.round(leaked, 3))

share of validation rows with an identical row in the training part, per fold: [0.392 0.386 0.394 0.392 0.404]


**What this cell does:** Cross-validates four models on the raw data (with duplicates) and on the de-duplicated data, and shows the gap.

**Why:** Shows how much each kind of model would have *looked* better than it really is if we'd kept the duplicates. The models span the range from memorising (full tree, random forest) to smooth (logistic regression, LDA).

In [4]:
models = {
    "full tree": DecisionTreeClassifier(random_state=config.SEED),
    "random forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=config.SEED),
    "logistic (scaled)": make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)),
    "LDA": LinearDiscriminantAnalysis(),
}
rows = []
for name, m in models.items():
    with_dups = evaluate.cross_validate(m, X_raw, y_raw, TASK, "", log=False)
    clean = evaluate.cross_validate(m, X, y, TASK, "", log=False)
    rows.append({"model": name, "CV with duplicates": with_dups.accuracy.mean(), "CV de-duplicated": clean.accuracy.mean(),
                 "inflation (points)": 100 * (with_dups.accuracy.mean() - clean.accuracy.mean()),
                 "train accuracy": clean.train_accuracy.mean()})
dup_table = pd.DataFrame(rows).set_index("model")
display(dup_table.round(4))

,CV with duplicates,CV de-duplicated,inflation (points),train accuracy
model,,,,
full tree,0.8957,0.8241,7.1567,1.0000
random forest,0.9292,0.8825,4.6667,1.0000
logistic (scaled),0.8957,0.8941,0.1567,0.8960
LDA,0.8930,0.8924,0.0600,0.8943


**Observations (Step 1)**

- **The leak is as large as predicted:** 39–40% of validation rows have an identical row in the training part, in every fold.
- **Inflation depends on how much the model memorises:**

  | Model | Train accuracy | CV with duplicates | CV de-duplicated | Inflation |
  |---|---|---|---|---|
  | Full tree | 1.000 | 0.896 | 0.824 | **+7.2 points** |
  | Random forest | 1.000 | 0.929 | 0.883 | **+4.7 points** |
  | Logistic | 0.896 | 0.896 | 0.894 | +0.2 |
  | LDA | 0.894 | 0.893 | 0.892 | +0.1 |

- **This would have changed our model choice.** With duplicates, the random forest (0.929) looks like the clear winner, 3.3 points ahead of logistic regression. De-duplicated, it's 1.2 points *behind*. Keeping the duplicates wouldn't just inflate the numbers; it would rank the models wrongly, in favour of exactly the models that memorise.
- The full tree with duplicates (0.896) even appears to tie logistic regression, although on genuinely unseen rows it's 7 points worse.
- **Rule for the report:** remove exact duplicates *before* splitting, so a row and its copy can never sit on both sides of a fold boundary.

## Step 2 — Standardisation: who needs it?

Standardising replaces each feature x by (x − mean) / std, computed on the training fold. Whether a model cares depends on how it uses the numbers:

| Model | How it uses feature values | Scale matters? |
|---|---|---|
| Decision tree / forest | only compares a value to a threshold | **No:** rescaling moves the threshold, same split |
| Gaussian NB | fits a mean and variance per feature | **No:** the fitted Gaussian rescales with the feature |
| LDA | uses the inverse covariance, which undoes any rescaling | **No** (in exact arithmetic) |
| **Logistic regression with a penalty** | penalty (1/2C) · Σ w² on raw weights | **Yes** |

Why logistic cares: to have the same effect on the prediction, a feature measured in kilograms (std 306) needs a weight ~7,000× smaller than one measured as a drag coefficient (std 0.04). The penalty sees only the weight sizes, so it barely touches the weight-in-kg coefficient but crushes the drag coefficient's, even if drag matters more. The penalty ends up depending on the choice of *units*, which is arbitrary.

There's also a practical problem: the optimiser (gradient-based) converges very slowly when features differ in scale by 10⁴.

**What this cell does:** Cross-validates each model with and without a StandardScaler in front, and for logistic regression also records the fitting time and whether the optimiser converged.

**Why:** Turns the table above into measured numbers, so the claim “only logistic needs scaling” is checked, not assumed.

In [5]:
def compare(name, model):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        t = time.time()
        raw_acc = evaluate.cross_validate(model, X, y, TASK, "", log=False).accuracy.mean()
        raw_time = time.time() - t
    not_converged = sum("converge" in str(x.message).lower() for x in w)
    scaled_acc = evaluate.cross_validate(make_pipeline(StandardScaler(), model), X, y, TASK, "", log=False).accuracy.mean()
    return {"model": name, "unscaled": raw_acc, "scaled": scaled_acc, "difference (points)": 100 * (scaled_acc - raw_acc),
            "unscaled fit time (s)": round(raw_time, 1), "folds not converged (unscaled)": not_converged}

scale_table = pd.DataFrame([
    compare("decision tree", DecisionTreeClassifier(random_state=config.SEED)),
    compare("Gaussian NB", GaussianNB()),
    compare("LDA", LinearDiscriminantAnalysis()),
    compare("logistic C=0.1 (max_iter=1000)", LogisticRegression(C=0.1, max_iter=1000)),
    compare("logistic C=0.01 (max_iter=1000)", LogisticRegression(C=0.01, max_iter=1000)),
]).set_index("model")
display(scale_table.round(4))

,unscaled,scaled,difference (points),unscaled fit time (s),folds not converged (unscaled)
model,,,,,
decision tree,0.8241,0.8247,0.0533,0.6,0
Gaussian NB,0.8465,0.8473,0.0800,0.2,0
LDA,0.8924,0.8924,0.0000,0.2,0
logistic C=0.1 (max_iter=1000),0.8755,0.8941,1.8667,6.4,5
logistic C=0.01 (max_iter=1000),0.8761,0.8909,1.4800,5.5,5


**What this cell does:** Fits unscaled and scaled logistic regression (C = 0.1) on all rows and compares, for each feature, the size of its effect: |weight| × feature std.

**Why:** Shows *how* the penalty distorts the unscaled model: which features it was allowed to use and which it suppressed.

In [6]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    unscaled = LogisticRegression(C=0.1, max_iter=1000).fit(X, y)
scaled = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)).fit(X, y)
effect = pd.DataFrame({
    "feature std": X.std(),
    "effect unscaled (|w|·std)": np.abs(unscaled.coef_).max(axis=0) * X.std().to_numpy(),
    "effect scaled (|w|)": np.abs(scaled[-1].coef_).max(axis=0),
}).sort_values("feature std")
display(effect.round(3))

,feature std,effect unscaled (|w|·std),effect scaled (|w|)
drag_coefficient,0.043,0.000,0.141
turning_radius_m,0.523,0.004,0.189
engine_displacement_l,0.611,0.001,0.560
engine_cylinders,1.502,0.006,0.105
acceleration_0_100_kmh_s,2.258,0.045,1.476
highway_fuel_efficiency_kmpl,2.499,0.159,0.365
city_fuel_efficiency_kmpl,2.555,0.137,0.272
braking_distance_m,7.231,0.472,0.477
fuel_tank_capacity_l,14.475,1.228,0.325
battery_capacity_kwh,16.936,1.981,0.661


**Observations (Step 2)**

- **Tree, NB and LDA: no meaningful change** (≤ 0.08 points; LDA exactly 0).
  - The tiny tree difference comes from ties: when two candidate splits are equally good, floating-point rounding of the scaled values can pick a different one.
  - NB's tiny difference comes from its `var_smoothing` term, which is a fraction of the *largest* feature variance, so it isn't fully scale-free.
- **Logistic regression without scaling loses 1.9 points** (0.876 vs 0.894 at C = 0.1), takes ~6 s instead of < 1 s, and **fails to converge in all 5 folds** even with 1,000 iterations.
- **The effect table shows why.** It compares each feature's actual influence on the prediction (|weight| × std):
  - **acceleration**, the strongest feature in the scaled model (1.48), is pushed down to **0.045** unscaled. Its std is only 2.3 s, so it needs a large raw weight, which the penalty punishes;
  - **drag coefficient** (std 0.04) is wiped out completely (0.000 vs 0.14);
  - features with big units (weight in kg, battery, fuel tank) keep large effects, because their raw weights are tiny and barely penalised.
  - So without scaling, the penalty decides which features matter **by their units**, not by their usefulness.
- **Rule:** standardise for any model with a penalty on weights (or that uses distances), inside the pipeline so each fold's scaler learns only from its own training rows. Trees, NB and LDA don't need it, but it does no harm.

## Step 3 — What we deliberately don't change

EDA found hard floors and ceilings (e.g. 23% of cars at exactly 110 mm ground clearance) and that 14 features are whole numbers. Options would be to un-clip, smooth, or add "is at floor" indicators. We leave the values as they are, because:

- **The test set has the same floors and the same rounding** (EDA 2.7). A transformation that helps on train but treats test values differently would create a mismatch.
- **The floor itself is informative:** 75% of the cars at the 110 mm floor are Sports Cars (next cell), so the clipped value carries real class signal.
- **Every model we use is fine with them:** trees split on them directly, and linear models treat them as ordinary values.
- **No outliers in the usual sense:** every value lies inside a plausible physical range (EDA 2.2), and no rows are extreme on many features at once.

**What this cell does:** Shows the class mix of rows sitting at the ground-clearance floor (110 mm) vs all rows.

**Why:** Checks the claim that the floor value itself carries class information.

In [7]:
at_floor = X.ground_clearance_mm == X.ground_clearance_mm.min()
display(pd.DataFrame({
    "at 110 mm floor": y[at_floor].map(pp.CLASS_NAMES).value_counts(normalize=True),
    "all rows": y.map(pp.CLASS_NAMES).value_counts(normalize=True),
}).round(3))
print(f"rows at floor: {at_floor.sum():,} of {len(X):,}")

,at 110 mm floor,all rows
vehicle_class,,
SUV,0.081,0.332
Sedan,0.171,0.339
Sports Car,0.748,0.329


rows at floor: 1,731 of 7,500


**Observations (Step 3)**

- **75% of the 1,731 cars at the 110 mm floor are Sports Cars** (vs 33% overall). The clipped value is one of the strongest single signals in the data, not a defect to repair.
- So floors, ceilings and rounding stay as they are. Train and test share them, and every model handles them.

## Summary

| Preprocessing step | Evidence | Applied |
|---|---|---|
| Drop 2,500 exact duplicates | Without it, CV is inflated by up to 7.2 points (full tree) and the **model ranking flips** (forest would look best) | once, before CV: `load_clean()` → 7,500 rows |
| Standardise features | Unscaled logistic loses 1.9 points, never converges, and its penalty suppresses features by their units (acceleration's effect 1.48 → 0.05); tree/NB/LDA unaffected | inside each logistic pipeline, re-fitted per fold |
| Keep floors / rounding | Same in test; 75% of floor rows are Sports Cars (signal, not error) | — |
| Missing values, categorical encoding | None exist | — |

The noise-robustness steps (training on noisy copies, noise-aware LDA) are part of the models and are covered in `03_models.ipynb`.